In [ ]:
# Instalación de librerías
!pip -q install numpy==1.26.4 pandas==2.2.2 spacy==3.7.5 unidecode==1.3.8 rapidfuzz==3.9.7
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 12.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Seleccionar opción "Runtime > Restart Session" antes de correr esta celda
import re
import ast
import json
import math
import unicodedata
from typing import Dict, List, Tuple, Set

import pandas as pd
import numpy as np
from unidecode import unidecode
from rapidfuzz import fuzz, process

import spacy
nlp = spacy.load("es_core_news_md")

pd.set_option("display.max_colwidth", 200)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

pdf_path = "/content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/Dataset.xlsx"

# El archivo tiene columnas: entrada, definicion_quechua, ejemplos
df_raw = pd.read_excel(pdf_path)

# Sanitizar encabezados por si vienen con espacios o BOM
df_raw.columns = [c.strip().lower() for c in df_raw.columns]

expected_cols = {"entrada", "definicion_quechua", "ejemplos"}
missing = expected_cols - set(df_raw.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

print(df_raw.head(3))
print("Filas:", len(df_raw))

Mounted at /content/drive
      entrada  \
0     Achalay   
1  Achanqaray   
2    Achikyay   

                                                                                                                 definicion_quechua  \
0  . Sumaq p’acha churakuypas, uyanchikta mayllakuspa llimp’ikuypas, q’apaqkunawan hawikuypas; wasipas imapas sumaqyachiy. Adornar.   
1                                                           . Watan watan payllamanta kawsaq hatun yuraqpas pukapas t’ika. Begonia.   
2       . Intipas ninapas sansapas, illapapas rirpupas, unupas, huk kawsaykunapa k’anchayninpas. Brillar, alumbrar, aclarar el día.   

                                                                                                                                                            ejemplos  
0                             [['Sipaskunaqa ñawraq llimp’i t’ikawanmi chukunkuta achalanku.', 'Las jóvenes adornan sus sombreros con diversos colores de flores.']]  
1  [['Ñawpaqqa sipaskunapas wa

In [ ]:
# Normalización ligera (sin perder glotal/apóstrofos)
def normalize_quz(text: str) -> str:
    import unicodedata
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize("NFC", text).strip()

import re

# --- trailers permitidos al final (casos y enclíticos frecuentes) ---
TRAIL = r"(?:ta|man|paq|pi|manta|wan|kama|rayku|hina|qa|ri|pas|taq|raq|ña|lla|pacha|mi|m|si|s|chá|chu){0,3}$"

# --- FUTURO por persona (formas normativas) ---
FUT_MAP = {
    "FUT_1SG":      rf"saq{TRAIL}",
    "FUT_2SG":      rf"nki{TRAIL}",
    "FUT_3SG":      rf"nqa{TRAIL}",
    "FUT_1PL.INCL": rf"su[-]?nchik{TRAIL}",
    "FUT_1PL.EXCL": rf"saq[-]?ku{TRAIL}",
    "FUT_2PL":      rf"nki[-]?chik{TRAIL}",
    "FUT_3PL":      rf"nqa[-]?ku{TRAIL}",
}
FUT_RE = {k: re.compile(v, re.I) for k, v in FUT_MAP.items()}

RE = {
  # --- Tiempo ---
  "TENSE=PST.EXP":   re.compile(rf"rqa{TRAIL}", re.I),
  "TENSE=PST.NEXP":  re.compile(rf"sqa{TRAIL}", re.I),
  "TENSE=FUT":       re.compile(rf"(?:saq|nqa|nki|su[-]?nchik|saq[-]?ku|nki[-]?chik|nqa[-]?ku){TRAIL}", re.I),

  # --- Aspecto / Modo ---
  # Escritura norm.: -chka (aceptamos alomorfía oral en matching pero normalizamos etiqueta a PROG)
  "ASPECT=PROG":     re.compile(rf"chka{TRAIL}", re.I),
  "MOOD=COND":       re.compile(rf"man{TRAIL}", re.I),
  "MOOD=DUB":        re.compile(rf"chus{TRAIL}", re.I),

  # Imperativos (formas normativas)
  "MOOD=IMP.2SG":      re.compile(rf"(?:^|[^a-zñáéíóú])y{TRAIL}", re.I),   # ¡-y!
  "MOOD=IMP.3SG":      re.compile(rf"chun{TRAIL}", re.I),                  # ¡-chun!
  "MOOD=IMP.1PL.INCL": re.compile(rf"su[-]?nchik{TRAIL}", re.I),           # ¡-su-nchik!
  "MOOD=IMP.1PL.EXCL": re.compile(rf"saq[-]?ku{TRAIL}", re.I),             # ¡-saq-ku!
  "MOOD=IMP.2PL":      re.compile(rf"y[-]?chik{TRAIL}", re.I),             # ¡-y-chik!
  # (Si en el futuro activas 3PL: chun[-]?ku{TRAIL})

  # --- Polaridad / Interrogación ---
  "POL=NEG":         re.compile(rf"(?:\bmana\b|\bama\b)[^$]*\bchu\b", re.I),   # mana/ama … -chu
  "Q":               re.compile(rf"chu{TRAIL}", re.I),                         # –chu (con o sin '?')

  # --- Persona/Número verbal "neutros" (evidencia para SUBJ=…) ---
  "SUBJ=1SG":        re.compile(rf"ni{TRAIL}", re.I),
  "SUBJ=2SG":        re.compile(rf"nki{TRAIL}", re.I),
  "SUBJ=3SG":        re.compile(rf"(?<![a-zñáéíóú])n{TRAIL}", re.I),
  "SUBJ=1PL.INCL":   re.compile(rf"nchik{TRAIL}", re.I),
  "SUBJ=1PL.EXCL":   re.compile(rf"yku{TRAIL}", re.I),
  "SUBJ=2PL":        re.compile(rf"nkichik{TRAIL}", re.I),  # forma compuesta
  "SUBJ=3PL":        re.compile(rf"nku{TRAIL}", re.I),

  # --- Nominal ---
  "N.NUM=PL":        re.compile(rf"kuna{TRAIL}", re.I),

  # Posesión robusta:
  # Plurales (fuertes) – admiten colas; descartan si hay morfemas verbales en el token
  "POSS=1PL.INCL":   re.compile(rf"nchik{TRAIL}", re.I),
  "POSS=1PL.EXCL":   re.compile(rf"yku{TRAIL}", re.I),
  "POSS=2PL":        re.compile(rf"yki[-]?chik{TRAIL}", re.I),
  "POSS=3PL":        re.compile(rf"nku{TRAIL}", re.I),

  # Singulares (ambigua con verbal): exigimos cola para favorecer nominal
  "POSS=1SG":        re.compile(rf"[a-zñ’]y{TRAIL}", re.I),
  "POSS=2SG":        re.compile(rf"yki{TRAIL}", re.I),
  "POSS=3SG":        re.compile(rf"(?<![a-zñáéíóú])n{TRAIL}", re.I),

  # --- Evidenciales (solo ATT y REP) ---
  "EVID=ATT":        re.compile(rf"(?:\bmi\b|\bm\b){TRAIL.replace('$','')}$", re.I),
  "EVID=REP":        re.compile(rf"(?:\bsi\b|\bs\b){TRAIL.replace('$','')}$", re.I),

  # Orden raíz + OBJ + (posibles T/A/M) + SUBJ; disparamos por cola distintiva del bloque OBJ
  "OBJ=1SG.SUBJ>2SG.OBJ": re.compile(rf"yki{TRAIL}", re.I),           # Ñuqa qu-yki
  "OBJ=2SG.SUBJ>1SG.OBJ": re.compile(rf"wa[-]?nki{TRAIL}", re.I),     # Qam qu-wa-nki
  "OBJ=3SG.SUBJ>1SG.OBJ": re.compile(rf"wa[-]?n{TRAIL}", re.I),       # Pay qu-wa-n
  "OBJ=1SG.SUBJ>3SG.OBJ": re.compile(rf"(?<![a-zñáéíóú])ni{TRAIL}", re.I),  # Ñuqa qu-ni
  "OBJ=3SG.SUBJ>2SG.OBJ": re.compile(rf"su[-]?nki{TRAIL}", re.I),     # Pay qu-su-nki
  "OBJ=2SG.SUBJ>3SG.OBJ": re.compile(rf"nki{TRAIL}", re.I),           # Qam qu-nki (resolveremos conflicto con wa-nki abajo)
}

# Heurística simple para saber si el token "parece" verbal
def token_has_verbal_markers(t: str) -> bool:
    return bool(re.search(r"(rqa|sqa|chka|man|saq|nqa|nki|chun|y[-]?chik)", t, re.I))

def quz_evidence(sent_quz: str) -> dict:
    s = normalize_quz(sent_quz)
    found = {}

    # Barrido general
    for tag, rx in RE.items():
        if rx.search(s):
            # Confianza por defecto
            conf = "alta"

            # Ambiguos: posesivos singulares solo “media”
            if tag in {"POSS=1SG","POSS=2SG","POSS=3SG"}:
                conf = "media"
            # Evidenciales: media
            if tag in {"EVID=ATT","EVID=REP"}:
                conf = "media"
            # OBJ: algunos pueden confundirse con otras terminaciones > media
            if tag.startswith("OBJ="):
                conf = "media" if tag not in {"OBJ=2SG.SUBJ>1SG.OBJ","OBJ=3SG.SUBJ>1SG.OBJ"} else "alta"

            found[tag] = conf

    # FUT por persona > añade TENSE=FUT y SUBJ=* inequívoco
    PERS_FROM_FUT = {
        "FUT_1SG": "SUBJ=1SG",
        "FUT_2SG": "SUBJ=2SG",
        "FUT_3SG": "SUBJ=3SG",
        "FUT_1PL.INCL": "SUBJ=1PL.INCL",
        "FUT_1PL.EXCL": "SUBJ=1PL.EXCL",
        "FUT_2PL": "SUBJ=2PL",
        "FUT_3PL": "SUBJ=3PL",
    }
    for ptag, rx in FUT_RE.items():
        if rx.search(s):
            found["TENSE=FUT"] = "alta"
            found[PERS_FROM_FUT[ptag]] = "alta"

    if "OBJ=2SG.SUBJ>1SG.OBJ" in found and "OBJ=2SG.SUBJ>3SG.OBJ" in found:
        del found["OBJ=2SG.SUBJ>3SG.OBJ"]

    return found

In [ ]:
def es_tags(oracion: str):
    """
    Devuelve:
      - tags_candidatas (set)
      - evidencias_es: [(etiqueta, fuente, confianza)]
    """
    if not isinstance(oracion, str):
        return set(), []
    doc = nlp(oracion)
    tags, ev = set(), []

    # Negación
    if any(tok.lemma_ == "no" and tok.dep_ in {"advmod","neg"} for tok in doc):
        tags.add("POL=NEG"); ev.append(("POL=NEG","es_parse","alta"))

    # Interrogación
    if "¿" in oracion and "?" in oracion:
        tags.add("Q"); ev.append(("Q","es_parse","alta"))

    # Marcadores dubitativos (indicio débil)
    if any(tok.lemma_ in {"quizá","quizás","tal","acaso"} for tok in doc):
        tags.add("MOOD=DUB"); ev.append(("MOOD=DUB","es_parse","baja"))  # (typo fijo)

    # Tiempo/modo por rasgos
    has_finite = False
    for tok in doc:
        if tok.pos_ in {"VERB","AUX"}:
            verbform = tok.morph.get("VerbForm")
            tense = tok.morph.get("Tense")
            mood  = tok.morph.get("Mood")
            if "Fin" in verbform:
                has_finite = True
            if "Imp" in mood:
                tags.add("MOOD=IMP"); ev.append(("MOOD=IMP","es_parse","media"))
            if "Fut" in tense:
                tags.add("TENSE=FUT"); ev.append(("TENSE=FUT","es_parse","media"))
            if "Past" in tense:
                tags.add("TENSE=PST.EXP"); ev.append(("TENSE=PST.EXP","es_parse","media"))

    # Presente por defecto si hay verbal finito y no se marcó tiempo
    if not any(t.startswith("TENSE=") for t in tags) and has_finite:
        tags.add("TENSE=PRES"); ev.append(("TENSE=PRES","es_parse","baja"))

    # Prohibición (heurística): neg + imperativo/subjuntivo cercano
    has_neg = "POL=NEG" in tags
    has_imp_or_subj = "MOOD=IMP" in tags or any("Sub" in t.morph.get("Mood") for t in doc if t.pos_ in {"VERB","AUX"})
    if has_neg and has_imp_or_subj:
        tags.add("POL=PROH"); ev.append(("POL=PROH","es_parse","baja"))

    # Plural nominal
    if any(tok.pos_=="NOUN" and tok.morph.get("Number")==["Plur"] for tok in doc):
        tags.add("N.NUM=PL"); ev.append(("N.NUM=PL","es_parse","baja"))

    # Posesivos (ES no define persona exacta > marcador débil)
    if any(tok.lemma_ in {"mi","mis","tu","tus","su","sus","nuestro","nuestra","nuestros","nuestras"} for tok in doc):
        tags.add("POSS=?"); ev.append(("POSS","es_parse","media"))

    # Pasado no experimentado (indicio) "había + participio"
    for i, tok in enumerate(doc):
        if tok.lemma_ == "haber" and tok.pos_ == "AUX" and "Past" in tok.morph.get("Tense"):
            if any(t.pos_ == "VERB" and "Part" in t.morph.get("VerbForm") for t in doc[i: i+5]):
                tags.add("TENSE=PST.NEXP?"); ev.append(("TENSE=PST.NEXP?","es_parse","media"))
                break

    return tags, ev

In [ ]:
# Normalizador canónico y aliases
CANONICAL = {
    "SUBJ=1SG","SUBJ=2SG","SUBJ=3SG","SUBJ=1PL.INCL","SUBJ=1PL.EXCL","SUBJ=2PL","SUBJ=3PL",
    "TENSE=PRES","TENSE=PST.EXP","TENSE=PST.NEXP","TENSE=FUT",
    "ASPECT=PROG",
    "MOOD=COND","MOOD=DUB","MOOD=IMP.2SG","MOOD=IMP.3SG","MOOD=IMP.1PL.INCL","MOOD=IMP.1PL.EXCL","MOOD=IMP.2PL",
    "POL=NEG","POL=PROH","Q",
    "N.NUM=PL","N.NUM=SG",
    "POSS=1SG","POSS=2SG","POSS=3SG","POSS=1PL.INCL","POSS=1PL.EXCL","POSS=2PL","POSS=3PL",
    "EVID=ATT","EVID=REP",
    # OBJ (solo 6)
    "OBJ=1SG.SUBJ>2SG.OBJ","OBJ=2SG.SUBJ>1SG.OBJ","OBJ=3SG.SUBJ>1SG.OBJ",
    "OBJ=1SG.SUBJ>3SG.OBJ","OBJ=3SG.SUBJ>2SG.OBJ","OBJ=2SG.SUBJ>3SG.OBJ",
}

ALIASES = {
    "1SG":"SUBJ=1SG","2SG":"SUBJ=2SG","3SG":"SUBJ=3SG","1PL.INCL":"SUBJ=1PL.INCL",
    "1PL.EXCL":"SUBJ=1PL.EXCL","2PL":"SUBJ=2PL","3PL":"SUBJ=3PL",
    "POSS=1":"POSS=1SG","POSS=2":"POSS=2SG","POSS=3":"POSS=3SG",
    "TENSE=PAST.EXP":"TENSE=PST.EXP","TENSE=PAST.NEXP":"TENSE=PST.NEXP",
    # descartar EVID=CONJ si aparece por herencia
}

def normalize_out(tags: set) -> set:
    out = set()
    for t in tags:
        t = ALIASES.get(t, t)
        if t in CANONICAL:
            out.add(t)
    # Regla: si hay SUBJ y no hay TENSE, asumir presente
    if any(tt.startswith("SUBJ=") for tt in out) and not any(tt.startswith("TENSE=") for tt in out):
        out.add("TENSE=PRES")
    return out

def fuse_tags(tags_es: set, evid_quz: dict) -> set:
    out = set()

    # Tiempo: QUZ prior
    if evid_quz.get("TENSE=PST.NEXP") == "alta":
        out.add("TENSE=PST.NEXP")
    elif evid_quz.get("TENSE=PST.EXP") == "alta":
        out.add("TENSE=PST.EXP")
    elif evid_quz.get("TENSE=FUT") == "alta":
        out.add("TENSE=FUT")
    else:
        if "TENSE=PST.EXP" in tags_es:
            out.add("TENSE=PST.EXP")
        elif "TENSE=FUT" in tags_es:
            out.add("TENSE=FUT")
        elif "TENSE=PRES" in tags_es:
            out.add("TENSE=PRES")

    # Aspecto / Modo
    if evid_quz.get("ASPECT=PROG"):
        out.add("ASPECT=PROG")
    if evid_quz.get("MOOD=COND"):
        out.add("MOOD=COND")
    if evid_quz.get("MOOD=DUB"):
        out.add("MOOD=DUB")
    for imp in ["MOOD=IMP.2SG","MOOD=IMP.3SG","MOOD=IMP.1PL.INCL","MOOD=IMP.1PL.EXCL","MOOD=IMP.2PL"]:
        if evid_quz.get(imp):
            out.add(imp)

    # Polaridad
    if evid_quz.get("POL=NEG"):
        if any(t.startswith("MOOD=IMP") for t in out):
            out.add("POL=PROH")
        else:
            out.add("POL=NEG")
    elif "POL=PROH" in tags_es:
        out.add("POL=NEG")
    elif "POL=NEG" in tags_es:
        out.add("POL=NEG")

    # Interrogación
    if evid_quz.get("Q") or "Q" in tags_es:
        out.add("Q")

    # Persona/Número SUBJ desde QUZ
    for p in ["SUBJ=1SG","SUBJ=2SG","SUBJ=3SG","SUBJ=1PL.INCL","SUBJ=1PL.EXCL","SUBJ=2PL","SUBJ=3PL"]:
        if evid_quz.get(p):
            out.add(p)

    # Número nominal / Posesión
    if evid_quz.get("N.NUM=PL"):
        out.add("N.NUM=PL")
    for poss in ["POSS=1SG","POSS=2SG","POSS=3SG","POSS=1PL.INCL","POSS=1PL.EXCL","POSS=2PL","POSS=3PL"]:
        if evid_quz.get(poss):
            out.add(poss)

    # Evidenciales (solo ATT y REP)
    for e in ("EVID=ATT","EVID=REP"):
        if evid_quz.get(e):
            out.add(e)

    # OBJ transitivo (solo 6)
    for obj in [
        "OBJ=1SG.SUBJ>2SG.OBJ","OBJ=2SG.SUBJ>1SG.OBJ","OBJ=3SG.SUBJ>1SG.OBJ",
        "OBJ=1SG.SUBJ>3SG.OBJ","OBJ=3SG.SUBJ>2SG.OBJ","OBJ=2SG.SUBJ>3SG.OBJ",
    ]:
        if evid_quz.get(obj):
            out.add(obj)

    return normalize_out(out)

In [ ]:
def _conf_num(conf_str: str) -> float:
    # mapeo simple: alta=1.0, media=0.66, baja=0.33
    return {"alta":1.0, "media":0.66, "baja":0.33}.get((conf_str or "").lower(), 0.33)

def make_row(rid: int, fuente: str, sent_es: str, sent_quz: str, nivel: str = "") -> dict:
    """
    Construye una fila con:
      id, fuente, sent_es, sent_quz, tags_candidatas, evidencias, tags_finales, nivel,
      confianza_global, notas
    """
    # 1) Tags desde ES
    tags_es, ev_es = es_tags(sent_es)

    # 2) Evidencia morfológica desde QUZ
    ev_quz = quz_evidence(sent_quz)

    # 3) Fusión en estándar
    tags_finales = sorted(list(fuse_tags(tags_es, ev_quz)))

    # 4) Candidatas = unión informativa de ES + QUZ (sin perder rastro de QUZ)
    tags_candidatas = sorted(list(set(tags_es) | set(ev_quz.keys())))

    # 5) Evidencias consolidadas
    evidencias = []
    evidencias.extend([(et, "es_parse", conf) for et, _, conf in ev_es])
    evidencias.extend([(et, "quz_regex", conf) for et, conf in ev_quz.items()])

    # 6) Confianza global: promedio de evidencias de las etiquetas que quedaron en tags_finales
    conf_vals = []
    by_tag = {}
    for et, src, conf in evidencias:
        if et not in by_tag:
            by_tag[et] = []
        by_tag[et].append(_conf_num(conf))

    for et in tags_finales:
        if et in by_tag:
            conf_vals.append(max(by_tag[et]))
    confianza_global = round(float(np.mean(conf_vals)) if conf_vals else 0.5, 2)

    return {
        "id": rid,
        "fuente": fuente,
        "sent_es": (sent_es or "").strip() if isinstance(sent_es, str) else "",
        "sent_quz": normalize_quz(sent_quz or ""),
        "tags_candidatas": tags_candidatas,
        "evidencias": evidencias,
        "tags_finales": tags_finales,
        "nivel": nivel,
        "confianza_global": confianza_global,
        "notas": ""
    }

In [ ]:
def parse_ejemplos(cell) -> List[Tuple[str,str]]:
    """
    Espera strings del tipo:
    "[['Oración QUZ', 'Oración ES'], ['Oración QUZ 2', 'Oración ES 2']]"
    Devuelve lista de (ES, QUZ)
    """
    if isinstance(cell, list):
        data = cell
    elif isinstance(cell, str):
        try:
            data = ast.literal_eval(cell)
        except Exception:
            return []
    else:
        return []

    pairs = []
    for item in data:
        if (isinstance(item, list) or isinstance(item, tuple)) and len(item) == 2:
            quz, es = item
            # Asegurar tipos
            quz = quz if isinstance(quz, str) else ""
            es  = es  if isinstance(es, str)  else ""
            pairs.append((es.strip(), normalize_quz(quz)))
    return pairs

def explode_rows(df_in: pd.DataFrame, fuente_tag: str = "MINEDU") -> pd.DataFrame:
    rows = []
    rid = 1
    for idx, r in df_in.iterrows():
        entrada = (r.get("entrada") or "").strip()
        defin   = normalize_quz(r.get("definicion_quechua") or "")
        ejemplos = parse_ejemplos(r.get("ejemplos"))

        # (opcional) fila para la definición como par (no hay ES paralelo) → omitir por ahora
        # Creamos filas por cada ejemplo (ES, QUZ)
        for es, quz in ejemplos:
            fuente = f"{fuente_tag}|entrada:{entrada}"
            rows.append(make_row(rid, fuente, es, quz, nivel=""))
            rid += 1

    return pd.DataFrame(rows)

df_pairs = explode_rows(df_raw)
df_pairs.head(5)

,id,fuente,sent_es,sent_quz,tags_candidatas,evidencias,tags_finales,nivel,confianza_global,notas
0,1,MINEDU|entrada:Achalay,Las jóvenes adornan sus sombreros con diversos colores de flores.,Sipaskunaqa ñawraq llimp’i t’ikawanmi chukunkuta achalanku.,"[N.NUM=PL, POSS=?, TENSE=PRES]","[(TENSE=PRES, es_parse, baja), (N.NUM=PL, es_parse, baja), (POSS, es_parse, media)]",[TENSE=PRES],,0.33,
1,2,MINEDU|entrada:Achanqaray,Antiguamente tanto las jóvenes como las señoras se adornaban con begonias.,Ñawpaqqa sipaskunapas warmikunapas achanqaray yurapa t’ikanwanmi achalakuqku.,"[N.NUM=PL, TENSE=PRES]","[(TENSE=PRES, es_parse, baja), (N.NUM=PL, es_parse, baja)]",[TENSE=PRES],,0.33,
2,3,MINEDU|entrada:Achikyay,"El viajero, viendo el brillo de las brasas, llegó a esa casa.","Ñan purikuq runaqa sansapa achikyayninta rikuspa tuta, chay wasiman chayasqa.","[N.NUM=PL, TENSE=PST.EXP]","[(TENSE=PST.EXP, es_parse, media), (N.NUM=PL, es_parse, baja)]",[TENSE=PST.EXP],,0.66,
3,4,MINEDU|entrada:Achira,Para la fiesta del Corpus Christi llevan tubérculos asados de achira al Cuzco.,Kurpus raymipaqqa Qusquman achira watyasqata apamunku.,"[N.NUM=PL, TENSE=PRES]","[(TENSE=PRES, es_parse, baja), (N.NUM=PL, es_parse, baja)]",[TENSE=PRES],,0.33,
4,5,MINEDU|entrada:Achka,En ese agujero dormían muchos pitos.,Achka hak’akllukunam chay t’uqupiqa puñusqa.,"[N.NUM=PL, TENSE=PRES]","[(TENSE=PRES, es_parse, baja), (N.NUM=PL, es_parse, baja)]",[TENSE=PRES],,0.33,


In [ ]:
out_csv = "/content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/annotated_es_quz.csv"
out_parquet = "/content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/annotated_es_quz.parquet"

# Serializar 'evidencias' como JSON para CSV
df_to_save = df_pairs.copy()
df_to_save["evidencias"] = df_to_save["evidencias"].apply(lambda x: json.dumps(x, ensure_ascii=False))

df_to_save.to_csv(out_csv, index=False, encoding="utf-8")
df_pairs.to_parquet(out_parquet, index=False)

print("Guardado:")
print("CSV  ->", out_csv)
print("PARQ ->", out_parquet)
print(df_pairs.head(3))

Guardado:
CSV  -> /content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/annotated_es_quz.csv
PARQ -> /content/drive/MyDrive/Dataset/Documentos_Oficiales_MINEDU/TextoExtraido/annotated_es_quz.parquet
   id                     fuente  \
0   1     MINEDU|entrada:Achalay   
1   2  MINEDU|entrada:Achanqaray   
2   3    MINEDU|entrada:Achikyay   

                                                                      sent_es  \
0           Las jóvenes adornan sus sombreros con diversos colores de flores.   
1  Antiguamente tanto las jóvenes como las señoras se adornaban con begonias.   
2               El viajero, viendo el brillo de las brasas, llegó a esa casa.   

                                                                        sent_quz  \
0                    Sipaskunaqa ñawraq llimp’i t’ikawanmi chukunkuta achalanku.   
1  Ñawpaqqa sipaskunapas warmikunapas achanqaray yurapa t’ikanwanmi achalakuqku.   
2  Ñan purikuq runaqa sansapa achikyayninta rikuspa tuta, ch

In [ ]:
from collections import Counter

def coverage(df: pd.DataFrame) -> pd.DataFrame:
    cand = Counter()
    finals = Counter()
    for tags in df["tags_candidatas"]:
        for t in tags:
            cand[t] += 1
    for tags in df["tags_finales"]:
        for t in tags:
            finals[t] += 1
    keys = sorted(set(list(cand.keys()) + list(finals.keys())))
    rows = []
    for k in keys:
        rows.append({"etiqueta": k, "candidatas": cand.get(k,0), "finales": finals.get(k,0)})
    return pd.DataFrame(rows).sort_values(by=["finales","candidatas","etiqueta"], ascending=[False, False, True])

cov = coverage(df_pairs)
cov.head(20)

,etiqueta,candidatas,finales
15,TENSE=PRES,1014,1014
16,TENSE=PST.EXP,225,225
4,POL=NEG,164,164
14,TENSE=FUT,91,73
10,Q,56,56
3,OBJ=3SG.SUBJ>1SG.OBJ,1,1
6,POSS=1PL.INCL,1,1
7,POSS=1SG,1,1
8,POSS=3PL,1,1
11,SUBJ=1PL.INCL,1,1


In [ ]:
data_demo = pd.DataFrame({
    "entrada": ["Achalay", "Achanqaray"],
    "definicion_quechua": [
        ". Sumaq p’acha churakuypas, uyanchikta mayllakuspa llimp’ikuypas, q’apaqkunawan hawikuypas; wasipas imapas sumaqyachiy. Adornar.",
        ". Watan watan payllamanta kawsaq hatun yuraqpas pukapas t’ika. Begonia."
    ],
    "ejemplos": [
        "[['Sipaskunaqa ñawraq llimp’i t’ikawanmi chukunkuta achalanku.', 'Las jóvenes adornan sus sombreros con diversos colores de flores.']]",
        "[['Ñawpaqqa sipaskunapas warmikunapas achanqaray yurapa t’ikanwanmi achalakuqku.', 'Antiguamente tanto las jóvenes como las señoras se adornaban con begonias.']]",
    ]
})

df_demo_pairs = explode_rows(data_demo, fuente_tag="DEMO")
print(df_demo_pairs[["sent_quz","sent_es","tags_candidatas","tags_finales","confianza_global"]])

                                                                        sent_quz  \
0                    Sipaskunaqa ñawraq llimp’i t’ikawanmi chukunkuta achalanku.   
1  Ñawpaqqa sipaskunapas warmikunapas achanqaray yurapa t’ikanwanmi achalakuqku.   

                                                                      sent_es  \
0           Las jóvenes adornan sus sombreros con diversos colores de flores.   
1  Antiguamente tanto las jóvenes como las señoras se adornaban con begonias.   

                  tags_candidatas  tags_finales  confianza_global  
0  [N.NUM=PL, POSS=?, TENSE=PRES]  [TENSE=PRES]              0.33  
1          [N.NUM=PL, TENSE=PRES]  [TENSE=PRES]              0.33  
